---
title: Interactive Demo
execute:
  enabled: true
---

This page's code cells are **executed live** by Quarto (via Jupyter) every time the
docs are built, and the charts below are real interactive Plotly figures — not
static images. Zoom, pan, and hover over them.

## Sample data

We load AAPL's bundled daily OHLCV dataset (shipped with qrt, no network
dependency) via `q.data.datasets.load`:

In [1]:
import pandas as pd
import plotly.io as pio

import qrt as q

# Renderer used only when THIS notebook is executed directly (e.g. via
# `make nb-execute`) so it has viewable output when opened in VS Code.
# "notebook_connected" loads plotly.js from a CDN link (keeps this .ipynb
# small) instead of embedding the full library inline. The *published* docs
# site does not depend on this at all: `execute: enabled: true` above makes
# Quarto always re-execute fresh with its own Jupyter engine, which extracts
# plotly.js into the site's own static demo_files/libs/ folder (no CDN, no
# per-chart embed) regardless of what's cached here.
# See https://github.com/orgs/quarto-dev/discussions/5264
#pio.renderers.default = "notebook_connected"

ohlc = q.data.datasets.load("aapl")
ohlc.tail()

,open,high,low,close,volume
datetime,,,,,
2026-07-10,314.720001,316.910004,312.170013,315.320007,34132300
2026-07-13,317.019989,323.450012,315.779999,317.309998,43257800
2026-07-14,313.760010,316.190002,311.910004,314.859985,36336800
2026-07-15,317.619995,328.730011,317.320007,327.500000,60957600
2026-07-16,328.010010,334.679993,326.790009,333.260010,62727600


## Feature engineering

A couple of `q.feat.qta` indicators on AAPL's close price. We compute the SMAs
over the full history (so there's no warm-up gap) then focus the charts below
on the last few years:

In [2]:
sma_20 = q.feat.qta.sma(ohlc["close"], 20)
sma_100 = q.feat.qta.sma(ohlc["close"], 100)

features = pd.DataFrame({"close": ohlc["close"], "sma_20": sma_20, "sma_100": sma_100}).loc["2022":]
features.tail()

,close,sma_20,sma_100
datetime,,,
2026-07-10,315.320007,298.103001,279.024360
2026-07-13,317.309998,299.187001,279.561089
2026-07-14,314.859985,300.373500,280.068622
2026-07-15,327.500000,301.927499,280.740221
2026-07-16,333.260010,303.628500,281.429456


## Interactive price chart

`q.plot.interactive.line` builds a Plotly figure with hover, zoom, and range-selector
buttons:

In [3]:
fig = q.plot.interactive.line(
    features,
    title="AAPL close price with SMA overlays",
    yaxis_title="Price",
)
fig.show()

## Interactive performance report

A simple SMA-crossover strategy on AAPL (long when `sma_20 > sma_100`) plotted as a
full equity / drawdown / stats tearsheet, benchmarked against buying and holding SPY:

In [4]:
spy = q.data.datasets.load("spy")

position = (sma_20 > sma_100).astype(int).shift(1).fillna(0)
strategy_returns = (ohlc["close"].pct_change() * position).rename("AAPL SMA crossover").loc["2022":]
benchmark_returns = spy["close"].pct_change().rename("SPY").loc["2022":]

fig = q.plot.interactive.performance(strategy_returns, benchmark=benchmark_returns)
fig.show()

## Monthly returns heatmap

In [5]:
fig = q.plot.interactive.monthly_heatmap(strategy_returns)
fig.show()